# Selective prediction

* **Figure 4** (`selection_results_1_aggregate.png`): selective accuracy vs.
  abstention rate, and confidence minus accuracy on the rejected and selected sets.
* **Appendix** (`selection_results_2_by_dataset.png`, `selection_results_3_by_family.png`).

Input: `results/selection.csv` (`scripts/experiments/run_selection.py`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from paper_utils import DATASETS_MAP, FIGURES_DIR, METHOD_COLOR, METHOD_LABEL, RESULTS_DIR, promote_semantic

plt.style.use('seaborn-v0_8')
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
INK  = plt.rcParams['axes.labelcolor']
GRID = plt.rcParams.get('grid.color', '#cccccc')

def style_axes(ax, xlabel=None, ylabel=None):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.grid(True, color=GRID, lw=0.7)
    ax.set_axisbelow(True)
    if xlabel is not None: ax.set_xlabel(xlabel, color=INK)
    if ylabel is not None: ax.set_ylabel(ylabel, color=INK)
    return ax

In [ ]:
METHODS = ['logprob', 'verbal_conf_stem6', 'basc_k2', 'basc_k5']

df = promote_semantic(pd.read_csv(RESULTS_DIR / 'selection.csv'))
df = df[df['method'].isin(METHODS)].copy()
print(f'rows={len(df)}, seeds={df["seed"].nunique()}, models={df["model"].nunique()}, '
      f'datasets={sorted(df["dataset"].unique())}, rates={sorted(df["rate"].unique())}')

QWEN_MODELS = {'Qwen/Qwen3-0.6B', 'Qwen/Qwen3-1.7B', 'Qwen/Qwen3-4B-Thinking-2507',
               'Qwen/Qwen3-8B', 'Qwen/Qwen3-14B'}
NON_QWEN_MODELS = set(df['model'].unique()) - QWEN_MODELS
MIN_RATE = 0.05   # rate 0 keeps everything

def combo_curve(metric, subset=None):
    """Mean and SEM of `metric` per (method, rate) across (model, dataset, seed) trials."""
    d = df if subset is None else df[subset]
    out = d.groupby(['method', 'rate'])[metric].agg(mean='mean', std='std', n='count').reset_index()
    out['sem'] = out['std'] / np.sqrt(out['n'].clip(lower=1))
    return out

def _plot_curve(ax, methods, metric, subset=None):
    curve = combo_curve(metric, subset)
    curve = curve[curve['rate'] >= MIN_RATE]
    for m in methods:
        sub = curve[curve['method'] == m].sort_values('rate')
        x, y, e = sub['rate'].to_numpy() * 100, sub['mean'].to_numpy(), sub['sem'].to_numpy()
        ax.plot(x, y, linestyle='-', color=METHOD_COLOR[m], lw=1.8, label=METHOD_LABEL[m])
        ax.fill_between(x, y - e, y + e, alpha=0.12, color=METHOD_COLOR[m], linewidth=0)

def _plot_gap(ax, methods, conf_col, acc_col):
    """Mean confidence minus accuracy (in %) of a slice, per abstention rate."""
    per = (df.assign(_gap=df[conf_col] - df[acc_col])
             .groupby(['method', 'rate'])['_gap'].agg(mean='mean', std='std', n='count').reset_index())
    per['sem'] = per['std'] / np.sqrt(per['n'].clip(lower=1))
    per = per[per['rate'] >= MIN_RATE]
    for m in methods:
        sub = per[per['method'] == m].sort_values('rate')
        x, y, e = sub['rate'].to_numpy() * 100, sub['mean'].to_numpy() * 100, sub['sem'].to_numpy() * 100
        ax.plot(x, y, color=METHOD_COLOR[m], lw=1.8, label=METHOD_LABEL[m])
        ax.fill_between(x, y - e, y + e, alpha=0.5, color=METHOD_COLOR[m], linewidth=0)
    ax.axhline(0, color=INK, lw=0.6, ls='--', alpha=0.6)

## Figure 4

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
ax = axes[0]
_plot_curve(ax, METHODS, 'sel_acc')
style_axes(ax, xlabel='Abstention rate (%)', ylabel='Selective accuracy')
ax.set_title('Selective Accuracy', color=INK)
ax.legend(loc='upper left', fontsize=9, frameon=False, labelcolor=INK, ncol=2)
ax = axes[1]
_plot_gap(ax, METHODS, 'rej_conf', 'rej_acc')
style_axes(ax, xlabel='Abstention rate (%)', ylabel='Conf. − Acc. (%)')
ax.set_title('Rejected Set', color=INK)
ax = axes[2]
_plot_gap(ax, METHODS, 'sel_conf', 'sel_acc')
style_axes(ax, xlabel='Abstention rate (%)', ylabel='Conf. − Acc. (%)')
ax.set_title('Selected Set', color=INK)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'selection_results_1_aggregate.png')
plt.show()

## Appendix: by dataset and by model family

In [ ]:
datasets = sorted(df['dataset'].unique())
fig, axes = plt.subplots(1, len(datasets), figsize=(13, 3), sharey=False)
for ax, ds in zip(axes, datasets):
    _plot_curve(ax, METHODS, 'sel_acc', subset=(df['dataset'] == ds))
    style_axes(ax, xlabel='Abstention rate (%)', ylabel='Selective accuracy' if ax is axes[0] else None)
    ax.set_title(DATASETS_MAP.get(ds, ds), color=INK)
axes[-1].legend(loc='upper left', fontsize=9, frameon=False, labelcolor=INK, ncol=1)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'selection_results_2_by_dataset.png')
plt.show()

In [ ]:
FAMILY_SUBSETS = {'Qwen': df['model'].isin(QWEN_MODELS), 'non-Qwen': df['model'].isin(NON_QWEN_MODELS)}
fig, axes = plt.subplots(1, len(FAMILY_SUBSETS), figsize=(10, 3), sharey=False)
for ax, (fam, subset) in zip(axes, FAMILY_SUBSETS.items()):
    _plot_curve(ax, METHODS, 'sel_acc', subset=subset)
    style_axes(ax, xlabel='Abstention rate (%)', ylabel='Selective accuracy' if ax is axes[0] else None)
    ax.set_title(f'{fam} ({df.loc[subset, "model"].nunique()} models)', color=INK)
axes[-1].legend(loc='upper left', fontsize=9, frameon=False, labelcolor=INK, ncol=2)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'selection_results_3_by_family.png')
plt.show()